In [1]:
import numpy as np
import copy

import kuhn_poker

## Kuhn Poker

In [2]:
table = kuhn_poker.KuhnPoker()

table.play("b")
table.play("c")

(True, 'JKbc', array([ 1, -1]))

In [15]:
table.start()
table.play("c")
table.play("c")

(True, 'JKcc', array([-1,  1]))

In [16]:
table.start()
table.play("c")
table.play("b")
table.play("c")

(True, 'JKcbc', array([-1,  1]))

In [18]:
table.start()
table.play("b")
table.play("b")

(True, 'JKbb', array([-2,  2]))

## Vanilla CFR

In [277]:
def vanilla_cfr(steps):
    hands = ["JQ", "JK", "QJ", "QK", "KJ", "KQ"]
    
    sigma = 0.5 * np.ones((2, 12, 2))
    sigma_sum = np.zeros((2, 12, 2))
    R = np.zeros((2, 12, 2))

    for step in range(steps):

        # Run over all the information sets
        u = np.zeros((2, 12, 2))
        for hand in hands:
            table = kuhn_poker.KuhnPoker(hand)
            full_tree_recursion(u, R, sigma, table.get_infoset(0)[0], np.array([1/6, 1/6]), table, sigma_sum)

        # \sigma update rule
        Rplus = np.maximum(R, 0)
        Rsum = np.sum(Rplus, axis=2, keepdims=True)
        sigma = np.where(Rsum > 0, Rplus / Rsum, 0.5)

    # Compute \hat{\sigma}
    totals = sigma_sum.sum(axis=2, keepdims=True)
    sigma_avg = np.where(totals > 0, sigma_sum / totals, 0.5)

    return sigma_avg, R, u

def full_tree_recursion(u, R, sigma, infoset, reach_proba, table, sigma_sum):
    player = table.plr_idx
    avg_utility = np.zeros(2)
    u_sigma_a = np.zeros((2, 2))

    # Update \hat{\sigma}
    sigma_sum[player, infoset, :] += reach_proba[player] * sigma[player, infoset, :]

    # For each action in the infoset
    for action in table.actions:
        a = 0 if action == "b" else 1
        action_prob = sigma[player, infoset, a]

        next_reach = reach_proba.copy()
        next_reach[player] *= action_prob

        next_table = copy.deepcopy(table)
        terminate, history, utility = next_table.play(action)

        if terminate:
            action_utility = utility
        else:
            next_infoset = next_table.get_infoset(next_table.plr_idx)[0]
            action_utility = full_tree_recursion(u, R, sigma, next_infoset, next_reach, next_table, sigma_sum)

        u[:, infoset, a] += reach_proba[1 - player] * action_utility
        u_sigma_a[:, a] = action_utility
        avg_utility += action_prob * action_utility

    # Compute the regret of the infoset for the player that is acting
    R[player, infoset, :] += reach_proba[1 - player] * (u_sigma_a[player, :] - avg_utility[player])

    return avg_utility

In [337]:
n_iters = 10_000
sigma, R, u = vanilla_cfr(n_iters)

/var/folders/c7/qw5x706d7yj1s562xjqm7k2m0000gn/T/ipykernel_85218/3727823622.py:19: RuntimeWarning: invalid value encountered in divide
  sigma = np.where(Rsum > 0, Rplus / Rsum, 0.5)
/var/folders/c7/qw5x706d7yj1s562xjqm7k2m0000gn/T/ipykernel_85218/3727823622.py:23: RuntimeWarning: invalid value encountered in divide
  sigma_avg = np.where(totals > 0, sigma_sum / totals, 0.5)


In [295]:
def get_infoset_index(infoset):
    card_values = {"J": 0, "Q": 1, "K": 2}
    action_idx = ["", "c", "b", "cb"].index(infoset[1:])
    card_idx = card_values[infoset[0]]
    
    return 3 * action_idx + card_idx

infosets_plr1 = ["J", "Q", "K", "Jcb", "Qcb", "Kcb"]
infosets_plr2 = ["Jc", "Qc", "Kc", "Jb", "Qb", "Kb"]

In [325]:
# Evaluate the utilities with the trained strategy

u_eval = np.zeros_like(u)
R_eval = np.zeros_like(R)
sigma_sum_eval = np.zeros_like(sigma)

value = np.zeros(2)
for hand in hands:
    table = kuhn_poker.KuhnPoker(hand)
    utilities = full_tree_recursion(u_eval, R_eval, sigma, table.get_infoset(0)[0], np.array([1/6, 1/6]), table, sigma_sum_eval)
    value += utilities *  1/6 # Weight by the prob of each hand

In [340]:
print(f"Results of Vanilla CFR on Kuhn Poker after {n_iters} iterations\n")
print(f"Average utility of Player 0: {round(value[0], 4)}")
print(f"Value of alpha: {round(sigma[0, get_infoset_index('J'), :][0], 4)}\n")

for infoset in infosets_plr1:
    print(f"{infoset:>3}: Bet (Call) {round(sigma[0, get_infoset_index(infoset), :][0], 4):>8} | Check (Fold) {round(sigma[0, get_infoset_index(infoset), :][1], 4)}")
print(" ")

for infoset in infosets_plr2:
    print(f" {infoset}: Bet (Call) {round(sigma[1, get_infoset_index(infoset), :][0], 4):>8} | Check (Fold) {round(sigma[1, get_infoset_index(infoset), :][1], 4)}")

Results of Vanilla CFR on Kuhn Poker after 10000 iterations

Average utility of Player 0: -0.0555
Value of alpha: 0.2078

  J: Bet (Call)   0.2078 | Check (Fold) 0.7922
  Q: Bet (Call)   0.0003 | Check (Fold) 0.9997
  K: Bet (Call)   0.6126 | Check (Fold) 0.3874
Jcb: Bet (Call)      0.0 | Check (Fold) 1.0
Qcb: Bet (Call)   0.5508 | Check (Fold) 0.4492
Kcb: Bet (Call)   0.9999 | Check (Fold) 0.0001
 
 Jc: Bet (Call)   0.3345 | Check (Fold) 0.6655
 Qc: Bet (Call)   0.0004 | Check (Fold) 0.9996
 Kc: Bet (Call)      1.0 | Check (Fold) 0.0001
 Jb: Bet (Call)   0.0001 | Check (Fold) 1.0
 Qb: Bet (Call)   0.3393 | Check (Fold) 0.6607
 Kb: Bet (Call)      1.0 | Check (Fold) 0.0001
